In [ ]:
import os

# path setting
current = os.getcwd()
while os.path.basename(current) != "Data_center_and_fossil_energy_Replication":
    current = os.path.dirname(current)

BASE_PATH = current

RAW = os.path.join(BASE_PATH, 'Data', 'raw')
TEMP = os.path.join(BASE_PATH, 'Data', 'temp')
USE = os.path.join(BASE_PATH, 'Data', 'use')
FIGURES = os.path.join(BASE_PATH, 'Results', 'Figures')
TABLES = os.path.join(BASE_PATH, 'Results', 'Tables')

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    os.makedirs(path, exist_ok=True)


In [ ]:
# Placebo test for exposure to cement plants only

import os
import pandas as pd
import numpy as np
from shapely.geometry import Point
import geopandas as gpd
from tqdm import tqdm
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def calculate_distance_vectorized(lat1, lon1, lat2_array, lon2_array):
    """Vectorized distance calculation using the Haversine formula."""
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2_array)
    lon2_rad = np.radians(lon2_array)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return c * 6371  # Earth radius in km

def parse_coordinates(coord_str):
    """Parse the Coordinates field (format: -8.5824000, 13.4052710)."""
    try:
        if pd.isna(coord_str):
            return None, None
        parts = str(coord_str).split(',')
        if len(parts) == 2:
            lat = float(parts[0].strip())
            lon = float(parts[1].strip())
            return lat, lon
        return None, None
    except:
        return None, None

def load_placebo_data(file_path, facility_type):
    """
    Load cement plant data with robust handling for Excel filter issues.
    
    Parameters:
    -----------
    file_path : str
        Path to the Excel file
    facility_type : str
        'cement'
    
    Returns:
    --------
    DataFrame with columns: latitude, longitude, year
    """
    print(f"\n📂 Loading {facility_type} data: {os.path.basename(file_path)}")
    
    df = None
    
    # Try multiple read methods
    try:
        # Method 1: standard read
        df = pd.read_excel(file_path, sheet_name='use', engine='openpyxl')
        print(f"  ✓ Read successful: {len(df):,} raw records")
        
    except ValueError as e:
        if "wildcard" in str(e):
            print(f"  ⚠️  Filter format issue detected, using fallback method...")
            
            try:
                # Method 2: read directly with openpyxl
                from openpyxl import load_workbook
                
                wb = load_workbook(file_path, data_only=True)
                ws = wb['use']
                
                # Read all rows
                data = []
                for row in ws.iter_rows(values_only=True):
                    data.append(row)
                
                # First row as header
                if len(data) > 0:
                    columns = data[0]
                    df = pd.DataFrame(data[1:], columns=columns)
                    print(f"  ✓ Fallback read successful: {len(df):,} raw records")
                else:
                    raise ValueError("Worksheet is empty")
                    
            except Exception as e2:
                print(f"  ❌ Fallback method also failed: {e2}")
                return None
        else:
            raise
            
    except Exception as e:
        print(f"  ❌ Read failed: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # If all methods failed
    if df is None:
        print(f"  ❌ All read methods failed")
        return None
    
    # Data processing
    try:
        # Parse Coordinates
        coords = df['Coordinates'].apply(parse_coordinates)
        df['latitude'] = coords.apply(lambda x: x[0])
        df['longitude'] = coords.apply(lambda x: x[1])
        
        # Process Start year
        if 'Start year' in df.columns:
            df['year'] = pd.to_numeric(df['Start year'], errors='coerce')
        else:
            print(f"  ⚠️  'Start year' not found, trying alternative year fields...")
            year_cols = [col for col in df.columns if 'year' in col.lower()]
            if year_cols:
                df['year'] = pd.to_numeric(df[year_cols[0]], errors='coerce')
                print(f"  ✓ Using field: {year_cols[0]}")
            else:
                raise ValueError("No year field found")
        
        # Keep valid records
        df_valid = df.dropna(subset=['latitude', 'longitude', 'year']).copy()
        df_valid['year'] = df_valid['year'].astype(int)
        
        print(f"  ✓ Valid records: {len(df_valid):,}")
        print(f"  ✓ Year range: {df_valid['year'].min()} - {df_valid['year'].max()}")
        print(f"  ✓ Latitude range: [{df_valid['latitude'].min():.2f}, {df_valid['latitude'].max():.2f}]")
        print(f"  ✓ Longitude range: [{df_valid['longitude'].min():.2f}, {df_valid['longitude'].max():.2f}]")
        
        return df_valid[['latitude', 'longitude', 'year']]
        
    except Exception as e:
        print(f"  ❌ Data processing failed: {e}")
        import traceback
        traceback.print_exc()
        return None

def match_gadm_attributes(df_coal, gadm_path):
    """Match GADM administrative attributes to coal plants by coordinates."""
    print("\n🗺️  Matching GADM administrative attributes...")
    
    try:
        gadm_gdf = gpd.read_file(gadm_path)
        print(f"  ✓ GADM data loaded: {len(gadm_gdf)} records")
    except Exception as e:
        print(f"  ❌ Failed to read GADM: {e}")
        return df_coal
    
    geometry = [Point(xy) for xy in zip(df_coal['Longitude'], df_coal['Latitude'])]
    coal_gdf = gpd.GeoDataFrame(df_coal, geometry=geometry, crs='EPSG:4326')
    
    if gadm_gdf.crs != coal_gdf.crs:
        gadm_gdf = gadm_gdf.to_crs(coal_gdf.crs)
    
    print("  - Running spatial join...")
    start_time = time.time()
    coal_matched = gpd.sjoin(coal_gdf, gadm_gdf, how='left', predicate='within')
    elapsed = time.time() - start_time
    print(f"  ✓ Spatial join complete, time used: {elapsed/60:.1f} min")
    
    matched_count = coal_matched['index_right'].notna().sum()
    print(f"  ✓ Matched: {matched_count:,}/{len(df_coal):,} records ({matched_count/len(df_coal)*100:.1f}%)")
    
    cols_to_drop = ['geometry', 'index_right']
    coal_matched = coal_matched.drop(columns=[c for c in cols_to_drop if c in coal_matched.columns])
    
    return pd.DataFrame(coal_matched)

def calculate_proximity_exposure(df_coal, facility_data, facility_name, buffer_distances, time_windows):
    """
    Calculate coal plant exposure to a given facility type.
    
    Parameters:
    -----------
    df_coal : DataFrame
        Coal plant data
    facility_data : DataFrame
        Facility data (latitude, longitude, year)
    facility_name : str
        Facility prefix ('ai', 'cement')
    buffer_distances : list
        Buffer radii
    time_windows : dict
        Time window definitions
    
    Returns:
    --------
    DataFrame with new exposure columns
    """
    
    print(f"\n🎯 Calculating {facility_name.upper()} exposure...")
    
    # Extract facility data
    fac_lat = facility_data['latitude'].values
    fac_lon = facility_data['longitude'].values
    fac_year = facility_data['year'].values.astype(int)
    
    # Create new variables
    new_columns = []
    for distance_km in buffer_distances:
        for window_name in time_windows.keys():
            col_name = f'{facility_name}_proximity_{distance_km}km_{window_name}'
            df_coal[col_name] = 0.0
            new_columns.append(col_name)
    
    # Add type decomposition for AI centers
    if facility_name == 'ai' and 'SECONDARY_PPTY_TYPE' in facility_data.columns:
        hyperscale_types = ['Hyperscale Data Center', 'Crypto Mining Data Center', 
                            'Wholesale Data Center', 'Cloud Data Center']
        fac_type = facility_data['SECONDARY_PPTY_TYPE'].fillna('other').values
        fac_is_hyperscale = np.isin(fac_type, hyperscale_types)
        
        for distance_km in buffer_distances:
            for window_name in time_windows.keys():
                col_name_hyper = f'{facility_name}_proximity_{distance_km}km_{window_name}_hyper'
                df_coal[col_name_hyper] = 0.0
                new_columns.append(col_name_hyper)
                
                col_name_other = f'{facility_name}_proximity_{distance_km}km_{window_name}_other'
                df_coal[col_name_other] = 0.0
                new_columns.append(col_name_other)
        
        has_type_decomposition = True
    else:
        has_type_decomposition = False
    
    print(f"  ✓ Created {len(new_columns)} exposure variables")
    
    # Calculate exposure measures
    print(f"  🚀 Starting calculation...")
    start_time = time.time()
    
    grouped = df_coal.groupby('GEM_unit_phase_ID')
    
    for gem_id, group_df in tqdm(grouped, desc=f"Processing {facility_name}", ncols=80):
        coal_lat = group_df['Latitude'].iloc[0]
        coal_lon = group_df['Longitude'].iloc[0]
        
        distances_km = calculate_distance_vectorized(coal_lat, coal_lon, fac_lat, fac_lon)
        
        for idx, row in group_df.iterrows():
            current_year = int(row['year'])
            
            for distance_km in buffer_distances:
                within_buffer = distances_km <= distance_km
                
                for window_name, (start_year, end_year) in time_windows.items():
                    
                    # Main variable
                    col_name = f'{facility_name}_proximity_{distance_km}km_{window_name}'
                    
                    if start_year is None:
                        valid_fac = within_buffer & (fac_year <= end_year) & (fac_year <= current_year)
                    else:
                        valid_fac = within_buffer & (fac_year >= start_year) & (fac_year <= end_year) & (fac_year <= current_year)
                    
                    if valid_fac.sum() > 0:
                        valid_distances_km = np.maximum(distances_km[valid_fac], 0.1)
                        # Weighted by 10 km: Σ(10 / distance_km)
                        df_coal.at[idx, col_name] = np.sum(10.0 / valid_distances_km)
                    else:
                        df_coal.at[idx, col_name] = 0.0
                    
                    # Type decomposition for AI centers
                    if has_type_decomposition:
                        # Hyperscale
                        col_name_hyper = f'{facility_name}_proximity_{distance_km}km_{window_name}_hyper'
                        valid_fac_hyper = valid_fac & fac_is_hyperscale
                        
                        if valid_fac_hyper.sum() > 0:
                            valid_distances_km = np.maximum(distances_km[valid_fac_hyper], 0.1)
                            df_coal.at[idx, col_name_hyper] = np.sum(10.0 / valid_distances_km)
                        else:
                            df_coal.at[idx, col_name_hyper] = 0.0
                        
                        # Other
                        col_name_other = f'{facility_name}_proximity_{distance_km}km_{window_name}_other'
                        valid_fac_other = valid_fac & (~fac_is_hyperscale)
                        
                        if valid_fac_other.sum() > 0:
                            valid_distances_km = np.maximum(distances_km[valid_fac_other], 0.1)
                            df_coal.at[idx, col_name_other] = np.sum(10.0 / valid_distances_km)
                        else:
                            df_coal.at[idx, col_name_other] = 0.0
    
    total_elapsed = time.time() - start_time
    print(f"  ✓ {facility_name.upper()} calculation complete! Time used: {total_elapsed/60:.1f} min")
    
    return df_coal, new_columns

def process_proximity_exposure():
    """Main function: calculate exposure to AI centers and cement plants."""
    
    # 1. Load coal plant data
    print("\n" + "="*80)
    print("📊 Step 1/6: Load coal plant data")
    print("="*80)
    
    coal_plants_path = os.path.join(TEMP, "gem_coal_plants_multi_record_sa_sinceoperating.dta")
    df_coal = pd.read_stata(coal_plants_path)
    print(f"✓ Coal plant records: {len(df_coal):,}")
    
    # 2. Match GADM attributes
    print("\n" + "="*80)
    print("📊 Step 2/6: Match GADM administrative attributes")
    print("="*80)
    
    gadm_path = os.path.join(RAW, "gadm_410.gpkg")
    df_coal = match_gadm_attributes(df_coal, gadm_path)
    
    # 3. Load AI center data
    print("\n" + "="*80)
    print("📊 Step 3/6: Load AI center data")
    print("="*80)
    
    ai_center_path = os.path.join(RAW, "SPGlobal_Export.xlsx")
    df_ai = pd.read_excel(ai_center_path, sheet_name='Sheet1')
    
    df_ai['YR_BUILT'] = pd.to_numeric(df_ai['YR_BUILT'], errors='coerce')
    
    if 'SECONDARY_PPTY_TYPE' not in df_ai.columns:
        df_ai['SECONDARY_PPTY_TYPE'] = 'other'
    
    df_ai_filtered = df_ai.dropna(subset=['LATITUDE', 'LONGITUDE', 'YR_BUILT']).copy()
    df_ai_filtered = df_ai_filtered.rename(columns={
        'LATITUDE': 'latitude',
        'LONGITUDE': 'longitude',
        'YR_BUILT': 'year'
    })
    df_ai_filtered['year'] = df_ai_filtered['year'].astype(int)
    
    print(f"✓ Valid AI centers: {len(df_ai_filtered):,}")
    
    # 4. Load cement plant data
    print("\n" + "="*80)
    print("📊 Step 4/6: Load cement plant data (Placebo)")
    print("="*80)
    
    cement_path = os.path.join(RAW, "Global-Cement-and-Concrete-Tracker_July-2025.xlsx")
    df_cement = load_placebo_data(cement_path, 'cement')
    
    # 5. Define parameters
    print("\n" + "="*80)
    print("📊 Step 5/6: Define exposure parameters")
    print("="*80)
    
    buffer_distances = [15, 25, 50]
    
    time_windows = {
        'before06': (None, 2005),
        '06_15': (2006, 2015),
        '16_19': (2016, 2019),
        '20_24': (2020, 2024)
    }
    
    print(f"✓ Buffer radii: {[f'{d}km' for d in buffer_distances]}")
    print(f"✓ Time windows: {list(time_windows.keys())}")
    print(f"✓ Distance weight unit: 10 km (i.e., Σ(10 / distance_km))")
    
    # 6. Calculate all exposure measures
    print("\n" + "="*80)
    print("📊 Step 6/6: Calculate all exposure measures")
    print("="*80)
    
    df_coal = df_coal.dropna(subset=['Latitude', 'Longitude', 'year'])
    
    all_new_columns = []
    
    # 6.1 AI center exposure
    df_coal, ai_cols = calculate_proximity_exposure(
        df_coal, df_ai_filtered, 'ai', buffer_distances, time_windows
    )
    all_new_columns.extend(ai_cols)
    
    # 6.2 Cement plant exposure (placebo)
    if df_cement is not None and len(df_cement) > 0:
        df_coal, cement_cols = calculate_proximity_exposure(
            df_coal, df_cement, 'cement', buffer_distances, time_windows
        )
        all_new_columns.extend(cement_cols)
    else:
        print("  ⚠️  Skip cement plant calculation (invalid data)")
        cement_cols = []
    
    # 7. Check AI center type decomposition
    print("\n" + "="*80)
    print("🔍 Check AI center type decomposition consistency")
    print("="*80)
    
    for distance_km in buffer_distances:
        print(f"\n{distance_km}km buffer:")
        for window_name in time_windows.keys():
            col_total = f'ai_proximity_{distance_km}km_{window_name}'
            col_hyper = f'ai_proximity_{distance_km}km_{window_name}_hyper'
            col_other = f'ai_proximity_{distance_km}km_{window_name}_other'
            
            if col_total in df_coal.columns and col_hyper in df_coal.columns:
                sum_parts = df_coal[col_hyper] + df_coal[col_other]
                diff = df_coal[col_total] - sum_parts
                max_diff = diff.abs().max()
                
                print(f"  {window_name}: max difference = {max_diff:.10f} {'✓' if max_diff < 1e-6 else '❌'}")
    
    # 8. Save results
    print("\n" + "="*80)
    print("💾 Save results")
    print("="*80)
    
    output_path = os.path.join(TEMP, "gem_coal_sa_prox_exp_with_placebo.dta")
    df_coal.to_stata(output_path, write_index=False, version=118)
    
    print(f"\n✅ Results saved to: {output_path}")
    print(f"✅ Total variables: {len(all_new_columns)}")
    print(f"   - AI centers: {len(ai_cols)}")
    if df_cement is not None and len(cement_cols) > 0:
        print(f"   - Cement plants: {len(cement_cols)}")
    
    return df_coal

def quick_check_results():
    """Quick check of the output file."""
    output_path = os.path.join(TEMP, "gem_coal_sa_prox_exp_with_placebo.dta")
    
    print("\n" + "="*80)
    print("📋 Check output file")
    print("="*80)
    
    try:
        df = pd.read_stata(output_path)
        
        ai_vars = [col for col in df.columns if col.startswith('ai_proximity_')]
        cement_vars = [col for col in df.columns if col.startswith('cement_proximity_')]
        
        print(f"\n✓ Total records: {len(df):,}")
        print(f"\n✓ AI center variables: {len(ai_vars)}")
        print(f"✓ Cement plant variables: {len(cement_vars)}")
        print(f"✓ Total exposure variables: {len(ai_vars) + len(cement_vars)}")
        
        # Show sample descriptive statistics
        print("\n" + "-"*80)
        print("Sample AI exposure stats (15km, before06):")
        if 'ai_proximity_15km_before06' in df.columns:
            col = 'ai_proximity_15km_before06'
            print(f"  Mean: {df[col].mean():.4f}")
            print(f"  Median: {df[col].median():.4f}")
            print(f"  Max: {df[col].max():.4f}")
            print(f"  Non-zero share: {(df[col] > 0).mean()*100:.2f}%")
        
        print("\n" + "-"*80)
        print("Sample cement exposure stats (15km, before06):")
        if 'cement_proximity_15km_before06' in df.columns:
            col = 'cement_proximity_15km_before06'
            print(f"  Mean: {df[col].mean():.4f}")
            print(f"  Median: {df[col].median():.4f}")
            print(f"  Max: {df[col].max():.4f}")
            print(f"  Non-zero share: {(df[col] > 0).mean()*100:.2f}%")
        
    except Exception as e:
        print(f"❌ Check failed: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    try:
        result_df = process_proximity_exposure()
        quick_check_results()
        
    except KeyboardInterrupt:
        print("\n⚠️ Execution interrupted by user")
    except Exception as e:
        import traceback
        traceback.print_exc()
